# Imports

In [1]:
import os
import sys
sys.path.append("..")

In [2]:
import re

In [ ]:
from AGoTI.model import ApiVLLMModel
import AGoTI.operations as ops
import AGoTI.goo as goo
import AGoTI.thoughts as got

# Model

In [4]:
API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
model_name = "DeepSeek V3"

In [5]:
model = ApiVLLMModel(API_KEY, API_URL, model_name)

# Graph

In [6]:
numbers = [5, 3, 1, 2]
max_n_candidates = 5
max_n_best = 4

In [7]:
def make_task_prompt(numbers, max_n_candidates=max_n_candidates):
    return [{
        "role": "user",
        "content": "# Task\n"
                "Using basic mathematical operations (+, -, *, /) "
                f"combine all {len(numbers)} numbers below so you have left with only 24. "
                "You can use each number only once. You may use numbers in any order. "
                "You can use the same operation several times\n"
                "Right now you are allowed to make only one move towards the goal "
                "(so, you can combine only 2 numbers)\n"
                f"Write {max_n_candidates} unique candidates for your move (each one on the new line)"
                "# Format\n"
                "Write your response in the following format:\n"
                "\"\"\"\n"
                "# Numbers: 1, 8, 1, 1\n"
                "> 1 + 1 = 2 (left: \\boxed{2, 8, 1})\n"
                "> 8 - 1 = 7 (left: \\boxed{1, 7, 1})\n"
                "...\n"
                "\"\"\"\n"
                "Follow the format strictly! Do not write anything else!\n"
                f"# Numbers: {numbers}\n"
    }]

In [8]:
class FirstMoveGenerator(ops.SimplePromptGenerator):
    def parse_generation(self, text):
        matches = re.findall(r"> (.+)", text)
        return matches

In [9]:
class MoveGenerator(ops.Generator):
    async def thought_collector(self):
        thoughts = ops.any_thought_waiter(self.subscribtions, self.parents)
        async for thought in thoughts:
            numbers = re.findall(r"\\boxed{(.+)}", thought.text)
            if numbers:
                try:
                    numbers = list(map(int, numbers[-1].split(", ")))
                except:
                    print("ERROR: generation parsing error!")
            yield ([thought], {"numbers": numbers})

    def make_prompt(self, numbers):
        return make_task_prompt(numbers)

    def parse_generation(self, text):
            matches = re.findall(r"> (.+)", text)
            return matches

In [10]:
class MoveScorer(ops.SimpleScoreGenerator):
    async def thought_collector(self):
        parents = ops.any_thought_waiter(self.subscribtions, self.parents)
        async for parent in parents:
            messages = parent.prompt.copy()
            messages.append({"role": "user", "content": parent.text})
            yield ([parent], {"messages": messages})

    def make_prompt(self, messages):
        messages.append({
                "role": "user",
                "content": "Considering the task above score the move "
                           "from 0.0 to 3.0 based on how likely it is to reach 24 with "
                           "numbers left, where\n"
                           "0.0 - it's impossible to reach 24 or move is incorrect.\n"
                           "1.0 - it's unlikely to reach 24.\n"
                           "2.0 - it's probable to reach 24.\n"
                           "3.0 - it's trivial to reach 24.\n"
                           "# Format\n"
                           "Write no more than 3 sentences!"
                           "In the end of response write score in the following format:\n"
                           "\"\"\"\n"
                           "> 1 + 1 = 2 (left: \\boxed{2, 8, 1})\n"
                           "Score: \\boxed{2}\n"
                           "\"\"\"\n"
                           "Follow the format carefully!"
            })
        return messages

There's a simplier way to achieve the same result without using iterations (generating & filtering), but this one is more general and educationally interesting.

In [11]:
root = FirstMoveGenerator(
    model,
    make_task_prompt(numbers),
    name="FirstMoveGenerator",
    description="Generates first move of the game"
    )
move_generator = MoveGenerator(
    model,
    status=ops.Status.ASLEEP,
    name="MoveGenerator",
    description="Generates next move of the game"
    )
move_scorer = MoveScorer(
    model,
    parents=[root, move_generator],
    name="MoveScorer",
    description="Independently scores last move of the game"
    )
filter_best = ops.NBestFilter(
    max_n_best,
    parents=[move_scorer],
    name="BestMoveFilter",
    description=f"Filters {max_n_best} best last moves"
    )
move_counter = ops.SimpleIterationCounter(
    parents=[filter_best.choices["True"]],
    name="MoveCounter",
    description="Counts moves"
)
iteration_filter = ops.IterationFilter(
    parents=[move_counter],
    threshold=len(numbers)-1,
    name="IterationFilter",
    description=f"Stops solution after {len(numbers)-1} moves"
)
iteration_filter.choices["Less"].add_children([move_generator])
filter_answer = ops.NBestFilter(
    1,
    parents=[iteration_filter.choices["GreaterOrEqual"]],
    name="AnswerFilter",
    description=f"Filters best answer"
    )

In [12]:
graph = goo.GraphOfOperations(
    [
        root,
        move_generator,
        move_scorer,
        filter_best,
        filter_best.choices["True"],
        filter_best.choices["False"],
        move_counter,
        iteration_filter,
        iteration_filter.choices["Less"],
        iteration_filter.choices["GreaterOrEqual"],
        filter_answer,
        filter_answer.choices["False"],
        filter_answer.choices["True"]
        ],
    [root],
    [filter_answer.choices["True"]]
)

# Inference

In [13]:
output = await graph.run()
output = output[0]
print(output)

6 * 4 = 24 (left: \boxed{24})


In [14]:
print(
    "Решение:",
    *list(map(lambda x: x.text, got.collect_branch(
    next(iter(iteration_filter.choices["GreaterOrEqual"].thoughts))
    ))),
    sep="\n"
    )

Решение:
3 * 2 = 6 (left: \boxed{5, 6, 1})
5 - 1 = 4 (left: \boxed{4, 6})
6 * 4 = 24 (left: \boxed{24})
